# Tiny ImageNet NC diagnostics: two independent T4 workers

Upload this notebook through **File > Import Notebook**. Select **GPU T4 x2**. Attach the source ZIP and a Tiny ImageNet-200 dataset with labeled validation data. For later sessions, also attach `session_resume_bundle.zip`.

All scientific execution occurs here on Kaggle. This notebook is delivered with no outputs. Allow many sessions; the 12-run queue is not expected to finish overnight. Do not change scientific configs after seeing results.

## Cell 1. Verify environment and two T4 GPUs

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil

assert Path('/kaggle/working').is_dir(), 'Run this notebook on Kaggle only.'
import torch
print('Python:', sys.version)
print('torch:', torch.__version__, 'CUDA:', torch.version.cuda)
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, 'Select GPU T4 x2.'
for i in range(2):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, round(props.total_memory / 1024**3, 2), 'GiB')
    assert 'T4' in props.name, 'The frozen hardware target is T4.'
print('Disk free GiB:', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))

## Cell 2. Locate the attached dataset

Keep the same Kaggle dataset version across sessions. Paste its public dataset URL/version below if available. Do not substitute an unlabeled test set.

In [ ]:
DATA_ROOT_OVERRIDE = None
DATASET_URL_AND_VERSION = ''  # Record the actual attached dataset URL/version.
ALLOW_DOWNLOAD = False       # True only with Kaggle Internet; uses /kaggle/working.

roots = []
for train in Path('/kaggle/input').rglob('train'):
    if train.is_dir() and (train.parent / 'val').is_dir():
        if len([p for p in train.iterdir() if p.is_dir()]) == 200:
            roots.append(train.parent)
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
elif len(roots) == 1:
    DATA_ROOT = roots[0]
elif len(roots) > 1:
    raise RuntimeError('Multiple datasets found. Set DATA_ROOT_OVERRIDE: ' + str(roots))
else:
    DATA_ROOT = None
    if not ALLOW_DOWNLOAD:
        raise FileNotFoundError('Add Input: attach tiny-imagenet-200, including train/ and val/val_annotations.txt or labeled val class folders.')
print('Dataset:', DATA_ROOT)

## Cell 3. Extract/copy the source bundle

Supports both the ZIP and Kaggle's automatically extracted dataset layout. It does not delete an existing working project.

In [ ]:
import zipfile, tempfile, hashlib

PROJECT = Path('/kaggle/working/neurocomputing_tinyimagenet_nc')
SOURCE_OVERRIDE = None  # Set only if several source versions are attached.

def valid_project(p):
    return all((p / name).is_file() for name in ['run_dual_gpu_scheduler.py', 'protocol_manifest.json', 'requirements_kaggle.txt'])

def copy_source(src):
    if PROJECT.exists() and any(PROJECT.iterdir()):
        for p in src.rglob('*'):
            if p.is_file():
                dest = PROJECT / p.relative_to(src)
                if not dest.is_file() or hashlib.sha256(dest.read_bytes()).digest() != hashlib.sha256(p.read_bytes()).digest():
                    raise RuntimeError('Existing project differs. Use a fresh notebook/session; outputs were not deleted.')
        print('Existing matching source retained.')
    else:
        shutil.copytree(src, PROJECT, dirs_exist_ok=True)

candidates = [p.parent for p in Path('/kaggle/input').rglob('run_dual_gpu_scheduler.py') if valid_project(p.parent)]
archives = list(Path('/kaggle/input').rglob('Neurocomputing_TinyImageNet_Kaggle_DualT4.zip'))
if SOURCE_OVERRIDE:
    candidates, archives = ([Path(SOURCE_OVERRIDE)], []) if Path(SOURCE_OVERRIDE).is_dir() else ([], [Path(SOURCE_OVERRIDE)])
if len(candidates) == 1 and not archives:
    copy_source(candidates[0])
elif not candidates and len(archives) == 1:
    with tempfile.TemporaryDirectory(dir='/kaggle/working', prefix='source_stage_') as tmp:
        stage = Path(tmp).resolve()
        with zipfile.ZipFile(archives[0]) as z:
            for info in z.infolist():
                if not (stage / info.filename).resolve().is_relative_to(stage):
                    raise ValueError('Unsafe ZIP member')
            assert z.testzip() is None, 'Corrupt source ZIP'
            z.extractall(stage)
        extracted = [p.parent for p in stage.rglob('run_dual_gpu_scheduler.py') if valid_project(p.parent)]
        assert len(extracted) == 1, 'Archive does not contain exactly one complete project'
        copy_source(extracted[0])
else:
    raise RuntimeError('Attach exactly one source version, or set SOURCE_OVERRIDE. Found: ' + str(candidates + archives))
os.chdir(PROJECT)
print('Project:', PROJECT)
print('Required files present:', valid_project(PROJECT))

## Cell 4. Install missing dependencies and restore a previous session

The script preserves Kaggle's preinstalled torch/torchvision pair. Restore verifies file checksums and rejects conflicting existing outputs. Keep both archives downloaded from every prior session.

In [ ]:
subprocess.run([sys.executable, 'scripts/install_missing.py'], check=True)
subprocess.run([sys.executable, 'scripts/restore_session.py'], check=True)
prepare = [sys.executable, 'scripts/prepare_dataset.py']
if DATA_ROOT is not None:
    prepare += ['--data-root', str(DATA_ROOT)]
if ALLOW_DOWNLOAD:
    prepare += ['--allow-download']
subprocess.run(prepare, check=True)
DATA_ROOT = Path(json.loads((PROJECT / 'outputs/dataset_source.json').read_text())['mounted_root'])
(PROJECT / 'outputs/user_dataset_source.json').write_text(json.dumps({'kaggle_url_and_version': DATASET_URL_AND_VERSION, 'mounted_root': str(DATA_ROOT)}, indent=2))
print('Dataset and three immutable splits ready:', DATA_ROOT)

## Cell 5. Scientific smoke and resume checks

All four methods are checked on small real-data subsets, including a task-2 pause/resume. These outputs stay in `smoke/`. Do not start the long queue if this cell fails.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests_or_checks', '-q'], check=True)
subprocess.run([sys.executable, 'scripts/run_smoke.py', '--data-root', str(DATA_ROOT)], check=True)
SMOKE_PASSED = True

## Cell 6. Read the frozen protocol and pending runs

In [ ]:
print((PROJECT / 'PROTOCOL.md').read_text())
subprocess.run([sys.executable, 'run_dual_gpu_scheduler.py', '--status'], check=True)

## Cell 7. Start/resume both GPUs in one cell

Each GPU runs a separate method/seed job. Default budget is 9.25 hours including a 15-minute checkpoint grace; archiving follows and needs extra time. If setup has already used substantial session time, REDUCE this session budget. This is an engineering wall-time control, not a change to training epochs.

The scheduler stops at committed epoch boundaries and automatically creates both session archives. Do not use a hard kernel restart to stop it. Ctrl-C requests a cooperative stop.

In [ ]:
assert globals().get('SMOKE_PASSED'), 'Finish Cell 5 successfully first.'
MAX_WALL_HOURS = 9.25
subprocess.run([sys.executable, 'run_dual_gpu_scheduler.py', '--max-wall-hours', str(MAX_WALL_HOURS), '--data-root', str(DATA_ROOT)], check=True)

## Cell 8. Inspect progress

Run this after Cell 7, or inspect `outputs/worker_logs/` in the file browser while Cell 7 runs. The notebook kernel cannot execute a second cell concurrently with a blocking training cell.

In [ ]:
subprocess.run([sys.executable, 'run_dual_gpu_scheduler.py', '--status'], check=True)
manifest_path = PROJECT / 'outputs/run_manifest.json'
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {'runs': []}
for row in manifest['runs']:
    print(row['run_id'], row['status'], 'task:', row.get('task'), 'epoch:', row.get('epoch'))
print('Failure reports:')
for p in sorted((PROJECT / 'outputs/failures').glob('*.json')):
    print(p.name)

## Cell 9. Optional final analysis

Leave RUN_FINAL_ANALYSIS=False during ordinary GPU sessions. Once all 12 runs complete, preferably run analysis in a separate CPU Kaggle session using the source plus results archive, preserving GPU time. All primary analyses, six metrics and prespecified horizons are generated. This can take substantial CPU time, especially with historical CIFAR bootstrap intervals.

In [ ]:
RUN_FINAL_ANALYSIS = False
if RUN_FINAL_ANALYSIS:
    completed = list((PROJECT / 'outputs').glob('tinyimagenet_*/completed.json'))
    assert len(completed) == 12, 'Finish all 12 runs before final analysis.'
    subprocess.run([sys.executable, 'analyze_temporal_diagnostics.py', '--include-legacy'], check=True)
else:
    print('Final analysis deferred; continue saving this session.')

## Cell 10. Create/refresh archives

In [ ]:
archives_ready = all((Path('/kaggle/working') / name).exists() for name in ['session_results_bundle.zip', 'session_resume_bundle.zip'])
if RUN_FINAL_ANALYSIS or not archives_ready:
    command = [sys.executable, 'create_archives.py']
    if RUN_FINAL_ANALYSIS:
        command += ['--final']
    subprocess.run(command, check=True)
else:
    print('Scheduler already created both archives. Proceed to download.')

## Cell 11. Download and confirm both ZIP files

Use Kaggle's **Output/files sidebar**, refresh it, then the download action beside each ZIP below. Browser automatic downloads and clickable notebook links are not reliable substitutes. Keep the session open until both files have finished downloading. Save a notebook version including outputs as an additional copy when available; persistence alone is not an external backup.

For the next session attach the same SOURCE package, SAME Tiny ImageNet dataset version, and this session's `session_resume_bundle.zip`. Cell 4 detects ZIP or automatically extracted resume input. Rename local downloads with a session/date so earlier backups are retained.

In [ ]:
import hashlib
for name in ['session_results_bundle.zip', 'session_resume_bundle.zip', 'FINAL_Neurocomputing_TinyImageNet_NC_Results.zip']:
    p = Path('/kaggle/working') / name
    if p.exists():
        print(name, 'size GiB:', round(p.stat().st_size / 1024**3, 3))
        print('Path:', p)
        checksum = p.with_suffix('.sha256.json')
        if checksum.exists():
            print(checksum.read_text())
print('Download BOTH session files from the Output sidebar before ending the session.')

## Cell 12. Re-list downloads if needed

This cell does not start training or regenerate large archives.

In [ ]:
for p in sorted(Path('/kaggle/working').glob('*.zip')):
    print(p.name, p.stat().st_size, 'bytes')
for p in sorted(Path('/kaggle/working').glob('*.sha256.json')):
    print(p.name)